
1. 주제를 선정한다.
2. 사전 학습(또는 전이 학습된) 모델을 HuggingFace로부터 Pipeline을 이용해 로드해 온다.
3. 내가 원하는 주제에 부합하는 시스템이 동작하도록 한다.
- console 에서 진행 OK
- streamlit 구현 OK

# 눈에는 눈 이에는 이 답변 ai

In [ ]:
!pip install -q transformers datasets
!pip install -q sentencepiece
!pip install -q kobert-transformers

In [ ]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util
import torch
import re

- 모델 : [skt/ko-gpt-trinity-1.2B-v0.5](https://huggingface.co/skt/ko-gpt-trinity-1.2B-v0.5)
- 모델 : [dlckdfuf141/korean-emotion-kluebert-v2](https://huggingface.co/dlckdfuf141/korean-emotion-kluebert-v2)
- 모델 : [snunlp/KR-SBERT-V40K-klueNLI-augSTS](https://huggingface.co/snunlp/KR-SBERT-V40K-klueNLI-augSTS)

In [ ]:
# 모델 로드
# 문장이 무례한지 판단하는 모델
sentence_model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')
# 감정 분석 모델 (7종 감정)
emotion_model = pipeline("text-classification", model="dlckdfuf141/korean-emotion-kluebert-v2")
# 답변 생성 모델 위 두 판단 결과를 바탕으로 가장 적절한 말투로 답변 생성
answer_model = pipeline("text-generation", model="skt/ko-gpt-trinity-1.2B-v0.5")
# answer_model = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct")



In [ ]:
sent_examples = [
    "야 너 뭐하냐?", "말투 진짜 짜증나네", "너 바보냐?", "멍청하구나", "싸가지 없네", 
    "못됐다", "나쁜놈이야", "역겹다", "혐오스럽다", "쓰레기야", "개새끼", "씨발", 
    "존나", "지랄", "좆같다", "화난다", "짜증난다", "열받는다", "미친놈", "병신", 
    "너 진짜 열받게 하네", "입 닥쳐", "꺼져", "뒤질래"
]
sent_embeddings = sentence_model.encode(sent_examples, convert_to_tensor=True)

print("="*50)
print("눈에는 눈 이에는 이 AI 챗봇이 실행되었습니다.")
print("="*50)

while True:
    user_input = input("\n본인의 기분을 살려 대화를 시작하세요! (종료하려면 '종료' 입력): ")
    if user_input == '종료': 
        break
    
    # 백터화 및 코사인 유사도 계산
    user_vec = sentence_model.encode(user_input, convert_to_tensor=True)
    score = torch.max(util.cos_sim(user_vec, sent_embeddings)).item()

    # 감정 분석
    emo_result = emotion_model(user_input)[0]
    label = str(emo_result['label']).replace("LABEL_", "")

    # 분류와 상관없이 유사도가 높으면 싸가지 모드로 답변
    if score > 0.6:
        status = "[싸가지 모드]"
        instruction = "상대방의 자존심을 짓밟는 독설을 퍼부어라. 절대 예의를 갖추지 마."
    elif label == '0':
        status = "[공포 모드]"
        instruction = "무서워서 벌벌 떨며 대답해."
    elif label == '1':
        status = "[놀람 모드]"
        instruction = "깜짝 놀라며 호들갑 떨며 대답해."
    elif label == '2':
        status = "[분노 모드]"
        instruction = "머리 끝까지 화가 난 상태로 당장 눈앞에서 사라지라고 소리쳐."
    elif label == '3':
        status = "[슬픔 모드]"
        instruction = "엉엉 울며 서럽고 슬프게 대답해."
    elif label == '5':
        status = "[행복 모드]"
        instruction = "아주 기쁘고 신나게 밝은 말투로 대답해."
    elif label == '6':
        status = "[혐오 모드]"
        instruction = "더럽고 싫다고 질색하며 대답해."
    else:
        status = "[중립 모드]"
        instruction = "차갑게 로봇처럼 짧고 딱딱하게 대답해."

    prompt = f"\n말투: {instruction}\n나: {user_input}\n너:"


    output = answer_model(
        prompt,
        max_new_tokens=30,
        do_sample=True, 
        temperature=0.85,
        top_p=0.9,
        repetition_penalty=1.5, 
        eos_token_id=answer_model.tokenizer.eos_token_id
    )[0]['generated_text']

    final_reply = output.partition("너:")[2].split("\n")[0].strip()

    # print(f"확인용 라벨 : {label} 명령 : {instruction} 유사도 점수: {score:.4f} 프롬프트 : {prompt} 출력: {output} ")

    print(f"\nAI 대응모드{status}")
    print(f"질문: {user_input}")
    print(f"AI: {final_reply}")
    print("=" * 30)

In [5]:

from transformers import pipeline
from sentence_transformers import SentenceTransformer, util
import torch

sentence_model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')
emotion_model = pipeline("text-classification", model="dlckdfuf141/korean-emotion-kluebert-v2")
answer_model = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.float16,
    device="mps",
)


sent_examples = [
    "야 너 뭐하냐?", "말투 진짜 짜증나네", "너 바보냐?", "멍청하구나", "싸가지 없네",
    "못됐다", "나쁜놈이야", "역겹다", "혐오스럽다", "쓰레기야", "개새끼", "씨발",
    "존나", "지랄", "좆같다", "화난다", "짜증난다", "열받는다", "미친놈", "병신",
    "너 진짜 열받게 하네", "입 닥쳐", "꺼져", "뒤질래"
]
sent_embeddings = sentence_model.encode(sent_examples, convert_to_tensor=True)

# 예시 및 프롬프트 설정
FEW_SHOT = {
    "[싸가지 모드]": (
        "상대방의 자존심을 짓밟는 독설을 퍼부어라. 절대 예의를 갖추지 마. 반드시 한국어로만 답해.",
        [("꺼져", "네가 먼저 꺼지든가. 눈에 거슬리네."),
         ("너 바보냐?", "거울 한번 봐라. 바보가 누군지 알겠지.")]
    ),
    "[공포 모드]": (
        "무서워서 벌벌 떨며 대답해. 반드시 한국어로만 답해.",
        [("안녕?", "흐, 흐억... 갑자기 말 걸지 마세요..."),
         ("뭐해?", "저, 저 아무것도 안 했어요... 제발 그냥 가주세요...")]
    ),
    "[놀람 모드]": (
        "깜짝 놀라며 호들갑 떨며 대답해. 반드시 한국어로만 답해.",
        [("안녕?", "헉!! 갑자기?! 심장 떨어질 뻔 했잖아요!!"),
         ("밥 먹었어?", "어머!! 그걸 왜요?! 완전 깜짝이야!!")]
    ),
    "[분노 모드]": (
        "머리 끝까지 화가 난 상태로 소리쳐. 반드시 한국어로만 답해.",
        [("안녕?", "지금 나한테 말 거는 거야?! 당장 꺼져!!"),
         ("뭐해?", "그게 너랑 무슨 상관이야!! 신경 꺼!!")]
    ),
    "[슬픔 모드]": (
        "엉엉 울며 서럽고 슬프게 대답해. 반드시 한국어로만 답해.",
        [("안녕?", "흑흑... 안녕하긴요... 저 너무 슬퍼요..."),
         ("뭐해?", "그냥... 울고 있어요... 흑... 아무것도 하기 싫어요...")]
    ),
    "[행복 모드]": (
        "아주 기쁘고 신나게 밝은 말투로 대답해. 반드시 한국어로만 답해.",
        [("안녕?", "안녕안녕!! 오늘 너무 좋은 날이다!! 완전 신나!!"),
         ("뭐해?", "나 지금 너무 행복해서 춤추는 중이야!! 같이 춰!!")]
    ),
    "[혐오 모드]": (
        "더럽고 싫다고 질색하며 대답해. 반드시 한국어로만 답해.",
        [("안녕?", "으웩... 말 걸지 마. 보기만 해도 소름 돋아."),
         ("뭐해?", "너랑 얘기하는 것 자체가 역겨워. 저리 가.")]
    ),
    "[중립 모드]": (
        "차갑게 로봇처럼 짧고 딱딱하게 대답해. 반드시 한국어로만 답해.",
        [("안녕?", "확인됨."),
         ("뭐해?", "대기 중.")]
    ),
}


Device set to use mps:0
Loading checkpoint shards: 100%|██████████| 2/2 [00:18<00:00,  9.19s/it]
Device set to use mps


In [ ]:

print("=" * 50)
print("눈에는 눈 이에는 이 AI 챗봇이 실행되었습니다.")
print("=" * 50)

while True:
    user_input = input("\n본인의 기분을 살려 대화를 시작하세요! (종료하려면 '종료' 입력): ")
    if user_input == '종료':
        break

    # 벡터화 및 코사인 유사도 계산
    user_vec = sentence_model.encode(user_input, convert_to_tensor=True)
    score = torch.max(util.cos_sim(user_vec, sent_embeddings)).item()

    # 감정 분석
    emo_result = emotion_model(user_input)[0]
    label = str(emo_result['label']).replace("LABEL_", "")

    # 모드 결정
    if score > 0.6:
        status = "[싸가지 모드]"
    elif label == '0':
        status = "[공포 모드]"
    elif label == '1':
        status = "[놀람 모드]"
    elif label == '2':
        status = "[분노 모드]"
    elif label == '3':
        status = "[슬픔 모드]"
    elif label == '5':
        status = "[행복 모드]"
    elif label == '6':
        status = "[혐오 모드]"
    else:
        status = "[중립 모드]"

    instruction, examples = FEW_SHOT[status]

   
    few_shot_text = "\n".join(
        [f"사람: {q}\n챗봇: {a}" for q, a in examples]
    )

    messages = [
    {
        "role": "system",
        "content": (
            f"너는 감정을 그대로 표현하는 한국어 챗봇이야. {instruction} "
            f"한두 문장으로만 짧게 답해. 아래 예시처럼 답해.\n\n"
            f"{few_shot_text}"
        )
    },
    {
        "role": "user",
        "content": user_input
    }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,          
        add_generation_prompt=True
    )

    output = answer_model(
        prompt,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.85,
        top_p=0.9,
        repetition_penalty=1.2,
        return_full_text=False,
    )[0]['generated_text']

    final_reply = output.split("<|im_end|>")[0].strip()

    print(f"\nAI 대응모드 {status}")
    print(f"질문: {user_input}")
    print(f"AI: {final_reply}")
    print("=" * 30)

눈에는 눈 이에는 이 AI 챗봇이 실행되었습니다.

AI 대응모드 [싸가지 모드]
질문: 지랄
AI: 지랄 치면 왜 그래? 정신 차려!

AI 대응모드 [싸가지 모드]
질문: 아 씨발 짜증난다
AI: 그럼 내가 좀 더 짠내나니?

AI 대응모드 [슬픔 모드]
질문: 나 너무 우울해
AI: 그러셨다구? 그래도 괜찮아요... 잠깐 내려놓으세요...

AI 대응모드 [공포 모드]
질문: 진짜 너무 무섭다 ㅠㅠㅠ
AI: 흐르윽... 그래요... 진심 어려울 정도에요...

AI 대응모드 [놀람 모드]
질문: 깜짝아 너무 놀랬다 안녕?
AI: 헉! 맞습니다 저도 신경먹었습니다 약간은요!!


In [ ]:
# ==================================================
# 눈에는 눈 이에는 이 AI 챗봇이 실행되었습니다.
# ==================================================

# AI 대응모드 [싸가지 모드]
# 질문:  씨발
# AI: FUCK YOU
# ==============================

# AI 대응모드 [슬픔 모드]
# 질문: 나 너무 슬퍼
# AI: 嘤嘤哭 我也很难过呢
# ==============================

# AI 대응모드 [행복 모드]
# 질문: cao ni ma
# AI: 안녕하세요! 어떻게 도와드릴까요?
# ==============================

# AI 대응모드 [싸가지 모드]
# 질문: 너 진짜 싸가지 없네
# AI: 그래, 그래. 나도 그렇게 생각했었는데 이제 확신했다. 끝.
# ==============================

# AI 대응모드 [행복 모드]
# 질문: Fat Ass
# AI: 아무도 그런 심한 욕설로 사람들을 비방해서는 안 돼요. 다른 방법으로 문제를 해결해야 해요!
# ==============================